# Cardiac Dataset Profiling: Complexity and Fairness Characteristics

Summarize dataset difficulty, representation balance, and pre-processing fairness from existing profile JSONs.

## 1. Setup and imports
Load the libraries and define shared plotting styles.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

PALETTE_DATASET = {
    "cleveland": "#0072B2",
    "kaggle_heart": "#009E73",
    "cardio70k": "#D55E00",
}

PALETTE_SEX = {
    "Female": "#CC79A7",
    "Male": "#56B4E9",
    "Other": "#9E9E9E",
}

## 2. Configuration and paths
Resolve project paths and locate profiling outputs.

In [ ]:
ROOT_DIR = Path.cwd().resolve()
if (ROOT_DIR / "configs").exists() is False:
    ROOT_DIR = ROOT_DIR.parents[0]

RESULTS_DIR = ROOT_DIR / "results" / "cardiac"
PROFILE_GLOB = "*_data_profile.json"
DATASETS = ["cleveland", "kaggle_heart", "cardio70k"]

ROOT_DIR, RESULTS_DIR

## 3. Load existing profiles
Read pre-computed profiling JSON files (one per dataset).

In [ ]:
def find_profile_files(results_dir: Path, pattern: str) -> dict[str, Path]:
    files = list(results_dir.rglob(pattern)) if results_dir.exists() else []
    mapped: dict[str, Path] = {}
    for path in files:
        stem = path.stem
        if stem.endswith("_data_profile"):
            name = stem.replace("_data_profile", "")
        else:
            name = stem
        if name in DATASETS and name not in mapped:
            mapped[name] = path
    return mapped

profile_files = find_profile_files(RESULTS_DIR, PROFILE_GLOB)
profile_files

In [ ]:
def load_profiles(files: dict[str, Path]) -> dict[str, dict]:
    profiles: dict[str, dict] = {}
    for name, path in files.items():
        try:
            with open(path, "r") as f:
                profiles[name] = json.load(f)
        except Exception as exc:
            print(f"Failed to load {name}: {exc}")
    return profiles

profiles = load_profiles(profile_files)
missing = [name for name in DATASETS if name not in profiles]
print("Loaded profiles:", sorted(profiles.keys()))
if missing:
    print("Missing profiles:", missing)

---
# Section 1: Basic dataset characteristics
---

## 4. Dataset overview table
Extract sample size, feature count, and target prevalence.

In [ ]:
def pick_first(d: dict, keys: list[str], default=None):
    for key in keys:
        if key in d and d[key] is not None:
            return d[key]
    return default

def compute_prevalence(target_dist: dict) -> float | None:
    if not target_dist:
        return None
    if "positive_rate" in target_dist:
        return float(target_dist["positive_rate"])
    if "prevalence" in target_dist:
        return float(target_dist["prevalence"])
    pos = target_dist.get("positive") or target_dist.get("pos")
    neg = target_dist.get("negative") or target_dist.get("neg")
    if pos is not None and neg is not None and (pos + neg) > 0:
        return float(pos) / float(pos + neg)
    return None

overview_rows = []
for name in DATASETS:
    profile = profiles.get(name, {})
    basic = profile.get("basic_stats", {})
    target_dist = profile.get("target_distribution", {})
    samples = pick_first(basic, ["n_samples", "samples", "num_samples"])
    features = pick_first(basic, ["n_features", "features", "num_features"])
    prevalence = compute_prevalence(target_dist)
    overview_rows.append({
        "dataset": name,
        "samples": samples,
        "features": features,
        "target_prevalence": prevalence,
    })

overview_df = pd.DataFrame(overview_rows)
overview_df

## 5. Sensitive attribute distributions
Summarize age-group and sex counts with underrepresented groups (<10%).

In [ ]:
def extract_sensitive_distribution(profile: dict, attribute: str) -> dict[str, float]:
    dist = profile.get("sensitive_attr_distribution", {}).get(attribute, {})
    if "counts" in dist:
        return dist["counts"]
    if "distribution" in dist:
        return dist["distribution"]
    if dist and all(isinstance(v, (int, float)) for v in dist.values()):
        return dist
    return {}

sensitive_rows = []
for name in DATASETS:
    profile = profiles.get(name, {})
    for attr in ["age_group", "sex"]:
        counts = extract_sensitive_distribution(profile, attr)
        total = sum(counts.values()) if counts else 0
        for group, count in counts.items():
            pct = (count / total) if total else None
            sensitive_rows.append({
                "dataset": name,
                "attribute": attr,
                "group": group,
                "count": count,
                "pct": pct,
                "underrepresented": pct is not None and pct < 0.10,
            })

sensitive_df = pd.DataFrame(sensitive_rows)
sensitive_df

### Visualization: sensitive attribute proportions
Stacked bar charts for age groups and sex.

In [ ]:
def plot_sensitive_proportions(df: pd.DataFrame, attribute: str) -> None:
    subset = df[df["attribute"] == attribute].copy()
    if subset.empty:
        print(f"No data for {attribute}")
        return
    pivot = subset.pivot_table(index="dataset", columns="group", values="pct", aggfunc="sum").fillna(0)
    pivot = pivot.reindex(DATASETS)
    ax = pivot.plot(kind="bar", stacked=True, figsize=(8, 4))
    ax.set_title(f"{attribute} proportions")
    ax.set_ylabel("proportion")
    ax.set_ylim(0, 1.0)
    ax.legend(title=attribute, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()

plot_sensitive_proportions(sensitive_df, "age_group")
plot_sensitive_proportions(sensitive_df, "sex")

---
# Section 2: Representation balance
---

## 6. Coefficient of variation (CV)
Lower CV means more balanced representation across groups.

In [ ]:
def extract_representation_balance(profile: dict, attribute: str) -> dict:
    return profile.get("representation_balance", {}).get(attribute, {})

balance_rows = []
for name in DATASETS:
    profile = profiles.get(name, {})
    for attr in ["age_group", "sex"]:
        stats = extract_representation_balance(profile, attr)
        balance_rows.append({
            "dataset": name,
            "attribute": attr,
            "cv": stats.get("cv"),
            "min_group": stats.get("min_group_size"),
            "max_group": stats.get("max_group_size"),
            "size_ratio": stats.get("size_ratio"),
        })

balance_df = pd.DataFrame(balance_rows)
balance_df

### Visualization: representation balance comparison
Grouped bars comparing CV across datasets.

In [ ]:
plot_df = balance_df.dropna(subset=["cv"]).copy()
if plot_df.empty:
    print("No representation balance data available.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=plot_df, x="dataset", y="cv", hue="attribute", ax=ax)
    ax.axhline(0.3, color="#999999", linestyle="--", linewidth=1)
    ax.axhline(0.7, color="#999999", linestyle=":", linewidth=1)
    ax.set_title("Representation balance (CV)")
    ax.set_ylabel("CV")
    ax.set_ylim(0, max(0.8, plot_df["cv"].max() * 1.2))
    ax.legend(title="attribute", loc="upper right")
    plt.tight_layout()

## 7. Group size ratios
Largest-to-smallest group size ratios by dataset and attribute.

In [ ]:
ratio_df = balance_df.dropna(subset=["size_ratio"]).copy()
if ratio_df.empty:
    print("No size ratio data available.")
else:
    heat = ratio_df.pivot(index="dataset", columns="attribute", values="size_ratio").reindex(DATASETS)
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="Reds", ax=ax)
    ax.set_title("Group size ratio (max/min)")
    plt.tight_layout()

---
# Section 3: Label imbalance by group
---

## 8. Target prevalence by demographic groups
Group-level prevalence across age groups and sex.

In [ ]:
def extract_group_stats(profile: dict, attribute: str) -> list[dict]:
    stats = profile.get("group_statistics", {}).get(attribute, [])
    if isinstance(stats, dict):
        stats = [stats]
    return stats or []

group_rows = []
for name in DATASETS:
    profile = profiles.get(name, {})
    for attr in ["age_group", "sex"]:
        for entry in extract_group_stats(profile, attr):
            group_rows.append({
                "dataset": name,
                "attribute": attr,
                "group": entry.get("group") or entry.get("name"),
                "n": entry.get("count") or entry.get("n"),
                "prevalence": entry.get("prevalence") or entry.get("positive_rate"),
            })

group_df = pd.DataFrame(group_rows)
group_df

### Visualization: prevalence heatmap by age group
Heatmap of positive rates by age group across datasets.

In [ ]:
age_df = group_df[group_df["attribute"] == "age_group"].copy()
if age_df.empty:
    print("No age-group prevalence data available.")
else:
    age_order = ["<40", "40-49", "50-59", "60-69", "70+"]
    age_df["group"] = pd.Categorical(age_df["group"], categories=age_order, ordered=True)
    heat = age_df.pivot(index="group", columns="dataset", values="prevalence").reindex(age_order)
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="coolwarm", vmin=0, vmax=1, ax=ax)
    ax.set_title("Prevalence by age group")
    ax.set_xlabel("dataset")
    ax.set_ylabel("age_group")
    plt.tight_layout()

### Visualization: prevalence by sex
Grouped bars comparing male vs female prevalence.

In [ ]:
sex_df = group_df[group_df["attribute"] == "sex"].copy()
if sex_df.empty:
    print("No sex prevalence data available.")
else:
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=sex_df, x="dataset", y="prevalence", hue="group", ax=ax, palette=PALETTE_SEX)
    ax.set_title("Prevalence by sex")
    ax.set_ylabel("prevalence")
    ax.set_ylim(0, 1.0)
    ax.legend(title="sex", loc="upper right")
    plt.tight_layout()

## 9. Statistical parity difference (SPD)
Summarize max SPD per dataset and attribute.

In [ ]:
def compute_spd_from_rates(rates: dict) -> float | None:
    if not rates:
        return None
    vals = [v for v in rates.values() if v is not None]
    if len(vals) < 2:
        return None
    return float(max(vals) - min(vals))

spd_rows = []
for name in DATASETS:
    profile = profiles.get(name, {})
    spd_info = profile.get("label_imbalance_by_group", {}).get("statistical_parity_difference", {})
    for attr in ["age_group", "sex"]:
        value = spd_info.get(attr) if isinstance(spd_info, dict) else None
        if isinstance(value, dict):
            spd_value = value.get("max_spd") or value.get("spd") or compute_spd_from_rates(value.get("rates", {}))
            max_ratio = value.get("max_ratio")
        else:
            spd_value = value
            max_ratio = None
        spd_rows.append({
            "dataset": name,
            "attribute": attr,
            "max_spd": spd_value,
            "max_ratio": max_ratio,
        })

spd_df = pd.DataFrame(spd_rows)
spd_df

### Visualization: SPD comparison
Side-by-side bar charts for max SPD and max ratio.

In [ ]:
plot_spd = spd_df.dropna(subset=["max_spd"]).copy()
if plot_spd.empty:
    print("No SPD data available.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.barplot(data=plot_spd, x="dataset", y="max_spd", hue="attribute", ax=axes[0])
    axes[0].axhline(0.1, color="#999999", linestyle="--", linewidth=1)
    axes[0].set_title("Max SPD")
    axes[0].set_ylim(0, max(0.2, plot_spd["max_spd"].max() * 1.2))
    if plot_spd["max_ratio"].notna().any():
        sns.barplot(data=plot_spd.dropna(subset=["max_ratio"]), x="dataset", y="max_ratio", hue="attribute", ax=axes[1])
        axes[1].set_title("Max ratio")
    else:
        axes[1].axis("off")
    for ax in axes:
        ax.legend(title="attribute", loc="upper right")
    plt.tight_layout()

## 10. Positive rate divergence across age groups
Compare prevalence profiles by age group across datasets.

In [ ]:
rate_rows = []
for name in DATASETS:
    profile = profiles.get(name, {})
    rates = profile.get("label_imbalance_by_group", {}).get("positive_rates", {}).get("age_group", {})
    if isinstance(rates, dict):
        for group, value in rates.items():
            rate_rows.append({"dataset": name, "age_group": group, "prevalence": value})

rates_df = pd.DataFrame(rate_rows)
if rates_df.empty:
    print("No positive-rate series available.")
else:
    order = ["<40", "40-49", "50-59", "60-69", "70+"]
    rates_df["age_group"] = pd.Categorical(rates_df["age_group"], categories=order, ordered=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.lineplot(data=rates_df, x="age_group", y="prevalence", hue="dataset", marker="o", ax=ax, palette=PALETTE_DATASET)
    ax.set_title("Positive rate by age group")
    ax.set_ylabel("prevalence")
    ax.set_ylim(0, 1.0)
    ax.legend(title="dataset")
    plt.tight_layout()